# AIMO 3 Submission - Fixed Version

**Before running:** Go to Settings (right panel) → Accelerator → Select **GPU T4 x2** or **GPU P100**

In [ ]:
# Step 1: Check GPU availability FIRST
import torch

print("="*50)
print("GPU CHECK")
print("="*50)

if not torch.cuda.is_available():
    print("❌ NO GPU DETECTED!")
    print("")
    print("To fix this:")
    print("1. Click 'Settings' in the right panel")
    print("2. Under 'Accelerator', select 'GPU T4 x2' or 'GPU P100'")
    print("3. Restart the notebook")
    print("")
    raise RuntimeError("GPU required! Enable GPU in Settings → Accelerator")
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"✓ GPU detected: {gpu_name}")
    print(f"✓ GPU count: {gpu_count}")
    print(f"✓ GPU memory: {gpu_mem:.1f} GB")
    
    # Determine model size based on GPU
    if "H100" in gpu_name or gpu_mem > 70:
        MODEL_SIZE = "72B"
        print("\n→ H100 detected: Will use 72B model")
    elif "A100" in gpu_name or gpu_mem > 35:
        MODEL_SIZE = "32B"
        print("\n→ A100 detected: Will use 32B model")
    else:
        MODEL_SIZE = "7B"
        print("\n→ T4/P100 detected: Will use 7B model")

In [ ]:
# Step 2: Install dependencies
print("Installing vLLM (this takes ~2 minutes)...")
!pip install vllm transformers accelerate -q 2>&1 | tail -1
print("✓ Installation complete")

In [ ]:
# Step 3: Imports and config
import os
import re
import signal
import subprocess
import tempfile
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Optional

import pandas as pd
from tqdm import tqdm

IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

# Select model based on GPU
MODELS = {
    "7B": "Qwen/Qwen2.5-Math-7B-Instruct",
    "32B": "Qwen/Qwen2.5-32B-Instruct",
    "72B": "Qwen/Qwen2.5-Math-72B-Instruct",
}

@dataclass
class Config:
    model_id: str = MODELS.get(MODEL_SIZE, MODELS["7B"])
    num_samples: int = 32 if MODEL_SIZE == "7B" else 16
    num_generations: int = 4
    temperature: float = 0.7
    max_new_tokens: int = 2048
    timeout: int = 5

config = Config()
print(f"\nConfig:")
print(f"  Model: {config.model_id}")
print(f"  Samples: {config.num_samples}")
print(f"  Generations: {config.num_generations}")

In [ ]:
# Step 4: Code executor
class PythonREPL:
    def __init__(self, timeout=5):
        self.timeout = timeout
    
    def __call__(self, code):
        imports = "import math\nimport numpy as np\nimport sympy as sp\nfrom sympy import *\n"
        code = imports + code
        
        lines = code.strip().split("\n")
        if "print(" not in lines[-1] and "=" not in lines[-1]:
            lines[-1] = f"print({lines[-1]})"
        code = "\n".join(lines)
        
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(code)
            path = f.name
        
        try:
            result = subprocess.run(
                ["python3", path],
                capture_output=True, text=True, timeout=self.timeout
            )
            os.unlink(path)
            if result.returncode == 0:
                return True, result.stdout.strip()[:500]
            return False, result.stderr.strip()[-200:]
        except Exception as e:
            os.unlink(path)
            return False, str(e)

executor = PythonREPL(config.timeout)
print("✓ Code executor ready")

In [ ]:
# Step 5: Answer extraction
def extract_boxed(text):
    idx = text.rfind("\\boxed")
    if idx < 0:
        return None
    i, depth = idx, 0
    while i < len(text):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                break
        i += 1
    if depth != 0:
        return None
    return text[idx+7:i]

def to_int(text):
    if not text:
        return -1
    text = re.sub(r"[\\$,\s{}]", "", text)
    text = re.sub(r"\\?text\{[^}]*\}", "", text)
    try:
        return max(0, min(99999, round(float(text))))
    except:
        return -1

print("✓ Answer extraction ready")

In [ ]:
# Step 6: Load model
from vllm import LLM, SamplingParams

print(f"Loading {config.model_id}...")
print("This may take 2-5 minutes...")

llm = LLM(
    model=config.model_id,
    tensor_parallel_size=torch.cuda.device_count(),
    dtype="auto",
    trust_remote_code=True,
    gpu_memory_utilization=0.90,
    max_model_len=4096,
)

tokenizer = llm.get_tokenizer()
print(f"\n✓ Model loaded!")

In [ ]:
# Step 7: Solver
def solve(problem):
    prompt = f"""Solve this math problem step by step. Use Python code when helpful.
Put your final answer in \\boxed{{}}.

Problem: {problem}

Solution:"""
    
    sampling = SamplingParams(
        temperature=config.temperature,
        max_tokens=config.max_new_tokens,
        stop=["```output"],
        include_stop_str_in_output=True,
    )
    
    candidates = [prompt] * config.num_samples
    
    for _ in range(config.num_generations):
        outputs = llm.generate(candidates, sampling)
        new_candidates = []
        for o in outputs:
            text = o.prompt + o.outputs[0].text
            if text.rstrip().endswith("```output"):
                blocks = re.findall(r"```python(.*?)```", text, re.DOTALL)
                if blocks:
                    _, out = executor(blocks[-1])
                    text = text.rstrip() + f"\n{out}\n```\n"
            new_candidates.append(text)
        candidates = new_candidates
    
    answers = []
    for text in candidates:
        val = to_int(extract_boxed(text))
        if val >= 0:
            answers.append(val)
    
    if not answers:
        return 0
    return Counter(answers).most_common(1)[0][0]

print("✓ Solver ready")

In [ ]:
# Step 8: Main loop
if IS_SUBMISSION:
    import kaggle_evaluation.aimo_3_inference_server
    
    def predict(df):
        answer = solve(df["problem"].iloc[0])
        return pd.DataFrame({"id": df["id"], "answer": [answer]})
    
    server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    server.serve()
else:
    # Test on reference
    ref = pd.read_csv("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv")
    
    correct = 0
    for i, row in ref.iterrows():
        pred = solve(row["problem"])
        ok = pred == row["answer"]
        correct += ok
        print(f"{i+1}. True={row['answer']}, Pred={pred} {'✓' if ok else '✗'}")
    
    print(f"\nAccuracy: {correct}/{len(ref)} = {correct/len(ref):.0%}")